In [1]:
from qdrant_client import QdrantClient
import json

def query_and_filter_documents(client, collection_name, document_ids, output_file=None):
    """
    Query and filter documents from Qdrant with an option to save results to a JSON file.

    Args:
        client (QdrantClient): Initialized Qdrant client instance.
        collection_name (str): Name of the collection to query.
        document_ids (list): List of document IDs to retrieve.
        output_file (str, optional): File path to save the results as JSON.

    Returns:
        list: List of filtered and decoded document results.
    """
    results = []

    def decode_node_content(content):
        try:
            node_data = json.loads(content)
            if "text" in node_data and isinstance(node_data["text"], str):
                try:
                    node_data["text"] = json.loads(f'"{node_data["text"]}"')
                except json.JSONDecodeError:
                    pass
            return node_data
        except json.JSONDecodeError as e:
            print(f"Error decoding node content: {e}")
            return content

    def decode_payload(payload):
        decoded_payload = {}
        for key, value in payload.items():
            if isinstance(value, str):
                try:
                    if key == "_node_content":
                        decoded_payload[key] = decode_node_content(value)
                    else:
                        decoded_payload[key] = json.loads(f'"{value}"')
                except json.JSONDecodeError:
                    decoded_payload[key] = value
            else:
                decoded_payload[key] = value
        return decoded_payload

    def filter_fields(decoded_payload):
        filtered_result = {
            "doc_id": decoded_payload.get("doc_id"),
            "metadata": decoded_payload.get("_node_content", {}).get("metadata"),
            "text": decoded_payload.get("_node_content", {}).get("text"),
            "document_id": decoded_payload.get("document_id"),
            "language": decoded_payload.get("language"),
        }
        if isinstance(filtered_result["text"], str):
            try:
                filtered_result["text"] = json.loads(filtered_result["text"])
            except json.JSONDecodeError:
                pass
        return filtered_result

    # Loop through document_ids and fetch results
    for doc_id in document_ids:
        filter_criteria = {
            "must": [{"key": "document_id", "match": {"value": doc_id}}]
        }

        # Query Qdrant with the filter
        scrolled_results, _ = client.scroll(
            collection_name=collection_name,
            scroll_filter=filter_criteria,
            limit=10
        )

        # Process each result
        for result in scrolled_results:
            decoded_payload = decode_payload(result.payload)
            filtered_payload = filter_fields(decoded_payload)
            results.append(filtered_payload)

    # Save results to a JSON file if output_file is provided
    if output_file:
        with open(output_file, "w", encoding="utf-8") as file:
            json.dump(results, file, ensure_ascii=False, indent=2)

    return results

/Users/chirawatchitpakdee/python_project/ai-chatbot/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Usage Example
client = QdrantClient(
    url="http://localhost:6333",
    api_key="QdrantReaduumIzRt55pweKjX8o3hbmZsi4nRntN9bm54tU9juCGbHP0S6AOtuCA"
)

collection_name = "faq_data"
document_ids = ["faq_th_0", "faq_en_0"]
output_file = "output.json"

# Query and filter documents with output saved to a file
filtered_results = query_and_filter_documents(client, collection_name, document_ids, output_file=output_file)

# Print results
print(json.dumps(filtered_results, ensure_ascii=False, indent=2))

[
  {
    "doc_id": "faq_th_0",
    "metadata": {
      "source": "FAQ Dataset",
      "row_index": 0,
      "language": "Thai",
      "online_offline": "Offline"
    },
    "text": {
      "Question Example": "สามารถชำระบิลด้วยบัตรเครดิตได้หรือไม่",
      "Answer Example": "รับชำระด้วยเงินสดเท่านั้น",
      "Question Type": "Gift Card"
    },
    "document_id": "faq_th_0",
    "language": "Thai"
  },
  {
    "doc_id": "faq_en_0",
    "metadata": {
      "source": "FAQ Dataset",
      "row_index": 0,
      "language": "English",
      "online_offline": "Offline"
    },
    "text": {
      "Question Example": "Can I pay the bill with a credit card?",
      "Answer Example": "Only cash payment is accepted.",
      "Question Type": "Gift Card"
    },
    "document_id": "faq_en_0",
    "language": "English"
  }
]
